## TriKinetics DAM (Drosophila Activity Monitor) Data Pipeline from 10/3 and 10/8

In [55]:
import argparse
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [56]:
# One entry per 24 h run.
WINDOWS = {
 'W1': dict(start='2026-10-04 13:04', end='2026-10-05 13:04', exclude={}, # dark 24 hours
   labels=["IGMedYWM","IGLoYWF","IGHiYWM","IGMedYWF","IGMedOrRM","TGMedYWF","Control1YWF","TGHiOrRM",
           "IGLoOrRM","TGMedOrRF","IGHiOrRF","TGHiOrRF","TGMedOrRM","IGLoOrRF","TGLoOrRF","TGLoYWM",
           "Control1YWM","Control2YWM","TGMedYWM","Control1OrRM","IGLoYWM","IGHiYWF","TGLoYWM","TGLoOrRM",
           "IGMedOrRF","Control1OrRF","IGHiOrRM","Control2OrRF","Control2YWF","TGLoYWF","Control2OrRM","TGLoYWF"]), # two values repeat and two don't exist
 'W2': dict(start='2026-10-05 14:14', end='2026-10-06 14:14', exclude={},
   labels=["TGLoOrRM","TGLoOrRF","IGLoOrRM","IGLoYWF","Control1YWM","IGHiYWM","IGHiOrRF","TGMedOrRF",
           "Control1OrRM","TGHiOrRF","IGHiYWF","TGHiOrRM","IGHiYWM","IGMedOrRF","Control1YWF","Control1OrRF",
           "TGLoYWM","TGHiYWF","Control2OrRF","TGMedYWF","Control2YWF","TGMedOrRM","IGMedOrRM","Control2OrRM",
           "IGHiOrRM","TGMedYWM","IGMedYWM","IGMedYWF","IGLoOrRF","Control2YWM","TGLoYWF","IGLoYWM"]),
 'W3': dict(start='2026-10-06 15:45', end='2026-10-07 15:45', exclude={},
   labels=["TGHiYWF","TGMedYWM","IGLoYWM","IGHiOrRM","IGMedYWM","IGHiYWM","TGMedYWF","TGHiYWM",
           "TGMedOrRF","Control1OrRF","IGMedYWF","Control2OrRF","Control1OrRM","TGLoYWM","TGMedOrRM","IGLoOrRM",
           "Control1YWM","Control2OrRM","TGLoOrRM","Control1YWF","IGHiYWF","IGLoYWF","TGHiOrRM","IGLoOrRF",
           "TGLoYWF","Control2YWF","IGMedOrRM","IGHiOrRF","TGLoOrRF","TGHiOrRF","Control2YWM","IGMedOrRF"]),
'W4': dict(start='2026-10-07 17:19', end='2026-10-08 17:19', exclude={},
   labels=["TGLoOrRM", "IGHiYWM", "IGLoOrRF", "Control1OrRF", "TGHiYWF", "TGMedOrRF", "Control1YWM", "IGLoOrRM",
          "IGHiYWF", "TGMedYWM", "TGHiOrRF", "Control1YWF", "Control1OrRM", "IGMedYWM", "IGMedOrRF", "TGLoOrRF", 
          "TGLoYWM", "Control2YWM", "IGHiOrRM", "IGMedOrRM", "Control2OrRF", "IGMedYWF", "TGLoYWF", "TGHiOrRM",
          "IGLoYWF", "IGHiOrRF", "TGMedOrRM", "TGMedYWF", "Control2YWF", "TGHiYWM", "IGLoYWM", "Control2OrRM"]),
}

In [68]:
DATA_FILE = "DAM Monitor 1 Oct 3-8.txt"

OUTDIR = "dam_results"

In [69]:
ACTIVITY = "MT"            # 'MT' (moves, beam-to-beam) or 'CT' (beam entries)
SLEEP_MIN = 5              # minutes of inactivity that define sleep
BIN_MIN = 30               # bin width for the activity / sleep profiles
LIGHTS_ON_CLOCK = None     # e.g. "09:00" to override; None = auto from the light sensor
DAY_HOURS = 12             # length of the light phase (ZT0..ZT12 = day)
DEAD_HOURS = 6             # no movement for the last N h of a window -> flagged "possibly dead"
AUTO_EXCLUDE_DEAD = True
AUTO_EXCLUDE_DUPLICATE_LABELS = True   # tubes whose label appears >1x in a window (identity unknown)
MIN_VALID_FRAC = 0.80      # tubes with less valid-data coverage are excluded
CONTROL_GROUPS = ["Control1", "Control2"]   # pooled as "Control" in the comparisons
STAT_WINDOWS = None        # e.g. ['W2','W3','W4'] to restrict stats/figures to some windows (None = all)

### Constants / styling

In [70]:
LABEL_RE = re.compile(r"^(Control1|Control2|IG|TG)(Lo|Med|Hi)?(YW|OrR)([MF])$")
GROUP_ORDER = ["Control1", "Control2", "IG-Lo", "IG-Med", "IG-Hi", "TG-Lo", "TG-Med", "TG-Hi"]
TREAT_GROUPS = [g for g in GROUP_ORDER if g[:2] in ("IG", "TG")]
GROUP_COLORS = {
    "Control1": "#4d4d4d", "Control2": "#9a9a9a",
    "IG-Lo": "#9ecae1", "IG-Med": "#3b8ac4", "IG-Hi": "#08306b",
    "TG-Lo": "#fdbb84", "TG-Med": "#e6550d", "TG-Hi": "#7f2704",
}
STRATA = [("YW", "M"), ("YW", "F"), ("OrR", "M"), ("OrR", "F")]
STRAIN_NAME = {"YW": "yw", "OrR": "Oregon-R"}
SEX_NAME = {"M": "male", "F": "female"}
WIN_MARK = {"W1": "D", "W2": "o", "W3": "s", "W4": "^"}
WIN_COLORS = {"W1": "#7b6bb0", "W2": "#2a9d8f", "W3": "#e9a23b", "W4": "#c0504d"}
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
NIGHT = "#d9dce3"

METRICS = {  # key: (axis label, short title)
    "total_act": ("Total activity (moves / 24 h)", "Total activity"),
    "waking_act": ("Waking activity (moves / awake min)", "Waking activity"),
    "act_day": ("Day activity (moves)", "Day activity"),
    "act_night": ("Night activity (moves)", "Night activity"),
    "sleep_min": ("Total sleep (min / 24 h)", "Total sleep"),
    "sleep_day": ("Day sleep (min)", "Day sleep"),
    "sleep_night": ("Night sleep (min)", "Night sleep"),
    "n_bouts": ("Sleep bouts (n / 24 h)", "Sleep bouts"),
    "mean_bout": ("Mean bout length (min)", "Mean bout length"),
    "max_bout": ("Longest bout (min)", "Longest bout"),
    "lat_night": ("Sleep latency after lights-off (min)", "Night sleep latency"),
    "pos_left": ("Time in left half (D12, %)", "Left-half dwell"),
}

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 9.5, "text.color": INK,
    "axes.edgecolor": "#9a9a9a", "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 10.5, "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8.5, "figure.dpi": 100, "savefig.dpi": 200,
})
SEQ = LinearSegmentedColormap.from_list("seq", ["#f4f8fc", "#c6dbef", "#6baed6", "#2171b5", "#08306b"])

### Decode

In [71]:
def load_dam(path):
    """Return dict{dtype -> DataFrame[minute x tube(1..32)]}, light Series, raw summary."""
    cols = ["idx", "date", "time", "status", "extra", "mon", "tube", "typ", "unused", "light"] + list(range(1, 33))
    raw = pd.read_csv(path, sep="\t", header=None, names=cols, dtype={"typ": str}, skipinitialspace=True)
    raw["ts"] = pd.to_datetime(raw["date"].str.strip() + " " + raw["time"].str.strip(), format="%d %b %y %H:%M:%S")
    n_all = len(raw)
    ok = raw[raw["status"] == 1]
    data = {}
    for typ, g in ok.groupby("typ"):
        g = g.drop_duplicates("ts", keep="last").set_index("ts").sort_index()
        data[typ] = g[list(range(1, 33))].astype(float)
    # legacy 'Ct' rows with valid status (DAM2-style file) fall back to MT/CT
    if "MT" not in data and "Ct" in data:
        data["MT"] = data["Ct"]
        data.setdefault("CT", data["Ct"])
    light_src = ok[ok["typ"] == ("MT" if "MT" in data else ok["typ"].iloc[0])]
    light = light_src.drop_duplicates("ts", keep="last").set_index("ts")["light"].sort_index().astype(float)
    info = dict(n_rows=n_all, n_valid=len(ok), n_bad=n_all - len(ok),
                span=(raw["ts"].min(), raw["ts"].max()), types=sorted(data))
    return data, light, info


def estimate_lights_on(light):
    """Median clock time (min after midnight) of genuine 0->1 sensor transitions (inside contiguous data)."""
    full = light.reindex(pd.date_range(light.index.min(), light.index.max(), freq="min"))
    prev = full.shift(1)
    ons = full.index[(prev == 0) & (full == 1)]
    offs = full.index[(prev == 1) & (full == 0)]
    tod = lambda ix: np.array([t.hour * 60 + t.minute for t in ix])
    if len(ons) == 0:
        return None, ons, offs
    return float(np.median(tod(ons))), ons, offs

### Process analysis windows

In [72]:
def parse_label(lab):
    m = LABEL_RE.match(lab)
    if not m:
        return None
    t, dose, strain, sex = m.groups()
    if t.startswith("Control"):
        group, treat, dose = t, "Control", None
    else:
        group, treat = f"{t}-{dose}", t
    return dict(treatment=treat, dose=dose, strain=strain, sex=sex, group=group)


def sleep_from_act(act, thr):
    a = np.asarray(act, float)
    inactive = a == 0                      # NaN (missing) is never inactive -> gaps break bouts
    sleep = np.zeros(len(a), bool)
    bouts, i, n = [], 0, len(a)
    while i < n:
        if inactive[i]:
            j = i
            while j < n and inactive[j]:
                j += 1
            if j - i >= thr:
                sleep[i:j] = True
                bouts.append((i, j - i))
            i = j
        else:
            i += 1
    return sleep, bouts


def process_windows(data, light, lights_on_min):
    """Return flies DataFrame, bouts DataFrame, per-fly profile arrays, window objects."""
    flies, bouts_all, prof, wobj = [], [], {}, {}
    act_key = ACTIVITY if ACTIVITY in data else "MT"
    for wname, w in WINDOWS.items():
        idx = pd.date_range(w["start"], w["end"], freq="min", inclusive="left")
        act = data[act_key].reindex(idx)
        pn = data["Pn"].reindex(idx) if "Pn" in data else None
        d12 = data["D12"].reindex(idx) if "D12" in data else None
        lt = light.reindex(idx)
        tod = np.array([t.hour * 60 + t.minute for t in idx])
        zt = ((tod - lights_on_min) % 1440) / 60.0
        day = zt < DAY_HOURS
        has_both = lt.dropna().nunique() > 1
        regime = "LD" if has_both else ("DD" if lt.dropna().mean() < 0.5 else "LL")
        wobj[wname] = dict(idx=idx, zt=zt, light=lt, regime=regime, act=act, start=idx[0],
                           valid_frac=float(act.notna().any(axis=1).mean()))
        labels = w["labels"]
        if len(labels) != 32:
            print(f"  ! {wname}: {len(labels)} labels (expected 32)")
        cnt = pd.Series(labels).value_counts()
        dup = set(cnt[cnt > 1].index)
        # night start (first crossing of ZT=DAY_HOURS inside the window)
        cross = np.where((zt[1:] >= DAY_HOURS) & (zt[:-1] < DAY_HOURS))[0]
        n0 = int(cross[0] + 1) if len(cross) else None
        bin_idx = np.floor(zt * 60 / BIN_MIN).astype(int)
        nb = int(24 * 60 / BIN_MIN)
        for tube in range(1, 33):
            lab = labels[tube - 1] if tube <= len(labels) else ""
            meta = parse_label(lab) or dict(treatment=None, dose=None, strain=None, sex=None, group=None)
            a = act[tube].to_numpy()
            valid = ~np.isnan(a)
            sleep, bouts = sleep_from_act(a, SLEEP_MIN)
            awake = valid & ~sleep
            moved = np.where(valid & (a > 0))[0]
            last_move_h = (valid.sum() - 0) and ((len(a) - 1 - moved[-1]) / 60.0 if len(moved) else np.inf)
            blen = np.array([b[1] for b in bouts], float)
            bstart = np.array([b[0] for b in bouts], int)
            bday = day[bstart] if len(bstart) else np.array([], bool)
            lat = np.nan
            if n0 is not None:
                s_after = np.where(sleep[n0:])[0]
                lat = float(s_after[0]) if len(s_after) else np.nan
            rec = dict(
                fid=f"{wname}-T{tube:02d}", window=wname, tube=tube, label=lab, regime=regime, **meta,
                valid_frac=valid.mean(),
                total_act=np.nansum(a), act_day=np.nansum(a[day]), act_night=np.nansum(a[~day]),
                waking_act=np.nansum(a[awake]) / max(awake.sum(), 1),
                sleep_min=sleep.sum(), sleep_day=sleep[day].sum(), sleep_night=sleep[~day].sum(),
                n_bouts=len(bouts), n_bouts_day=int(bday.sum()), n_bouts_night=int((~bday).sum()),
                mean_bout=blen.mean() if len(blen) else np.nan, max_bout=blen.max() if len(blen) else 0,
                lat_night=lat, hours_since_last_move=last_move_h,
            )
            if d12 is not None:
                rec["pos_left"] = float(np.nanmean(d12[tube].to_numpy()))
            if pn is not None:
                p = pn[tube].to_numpy()
                p = p[~np.isnan(p)]
                p = p[(p >= 1) & (p <= 15)]
                rec["pos_mean"] = float(p.mean()) if len(p) else np.nan
                for b in range(1, 16):
                    rec[f"pn{b}"] = float((p == b).mean()) if len(p) else np.nan
            # ---- flags ----
            flags = []
            if meta["group"] is None:
                flags.append("unparsed label")
            if lab in dup and AUTO_EXCLUDE_DUPLICATE_LABELS:
                flags.append("duplicate label (identity ambiguous)")
            if tube in w["exclude"]:
                flags.append(f"manual: {w['exclude'][tube]}")
            if rec["valid_frac"] < MIN_VALID_FRAC:
                flags.append("low coverage")
            if np.isinf(last_move_h) or (last_move_h >= DEAD_HOURS):
                rec["possibly_dead"] = True
                if AUTO_EXCLUDE_DEAD:
                    flags.append(f"no movement for last {min(last_move_h, 24):.0f} h (dead?)")
            else:
                rec["possibly_dead"] = False
            rec["excluded"] = bool(flags)
            rec["flags"] = "; ".join(flags)
            flies.append(rec)
            # profiles (mean per BIN_MIN bin over valid minutes)
            cnt_b = np.bincount(bin_idx[valid], minlength=nb).astype(float)
            sa = np.bincount(bin_idx[valid], weights=a[valid], minlength=nb)
            ss = np.bincount(bin_idx[valid], weights=sleep[valid].astype(float), minlength=nb)
            with np.errstate(invalid="ignore", divide="ignore"):
                prof[rec["fid"]] = (sa / cnt_b * BIN_MIN, ss / cnt_b * BIN_MIN)
            for (s0, ln), isd in zip(bouts, bday):
                bouts_all.append(dict(fid=rec["fid"], window=wname, start_zt=zt[s0], length=ln, day=bool(isd)))
    fl = pd.DataFrame(flies)
    fl["stratum"] = fl["strain"].astype(str) + "-" + fl["sex"].astype(str)
    return fl, pd.DataFrame(bouts_all), prof, wobj

### Statistics

In [73]:
def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    order = np.argsort(pv)
    m = len(pv)
    adj = np.maximum.accumulate((m - np.arange(m)) * pv[order])
    res = np.empty(m)
    res[order] = np.minimum(adj, 1)
    out[ok] = res
    return out


def bh(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    adj = np.minimum.accumulate((pv[order] * m / (np.arange(m) + 1))[::-1])[::-1]
    res = np.empty(m)
    res[order] = np.minimum(adj, 1)
    out[ok] = res
    return out


def hedges_g(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    sp = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    if sp == 0:
        return np.nan
    return (x.mean() - y.mean()) / sp * (1 - 3 / (4 * (nx + ny) - 9))


def stratum_tests(fl, metrics):
    """Each treatment group vs pooled controls, inside each strain x sex stratum (Mann-Whitney U)."""
    rows = []
    for mk in metrics:
        for strat in ["all"] + [f"{a}-{b}" for a, b in STRATA]:
            sub = fl if strat == "all" else fl[fl["stratum"] == strat]
            ctrl = sub[sub["group"].isin(CONTROL_GROUPS)][mk].dropna()
            for g in TREAT_GROUPS:
                x = sub[sub["group"] == g][mk].dropna()
                p = np.nan
                if len(x) >= 2 and len(ctrl) >= 2:
                    p = stats.mannwhitneyu(x, ctrl, alternative="two-sided").pvalue
                rows.append(dict(metric=mk, stratum=strat, group=g, n=len(x), n_ctrl=len(ctrl),
                                 mean=x.mean() if len(x) else np.nan, ctrl_mean=ctrl.mean() if len(ctrl) else np.nan,
                                 pct_change=(x.mean() / ctrl.mean() - 1) * 100 if len(x) and ctrl.mean() else np.nan,
                                 hedges_g=hedges_g(x, ctrl), p_raw=p))
    df = pd.DataFrame(rows)
    df["p_fdr"] = np.nan
    for (mk, strat), idx in df.groupby(["metric", "stratum"]).groups.items():
        df.loc[idx, "p_fdr"] = bh(df.loc[idx, "p_raw"])
    return df


def ols_tests(fl, metrics):
    """Pooled OLS: metric ~ group(ref = Control) + strain + sex + window; Holm across treatment groups.
       Plus Type-II ANOVA with group x strain and group x sex interactions."""
    try:
        import statsmodels.formula.api as smf
        import statsmodels.api as sm
    except Exception:
        print("  (statsmodels not installed: skipping OLS / ANOVA)")
        return pd.DataFrame(), pd.DataFrame()
    d = fl.copy()
    d["grp"] = d["group"].where(~d["group"].isin(CONTROL_GROUPS), "Control")
    coefs, anovas = [], []
    for mk in metrics:
        dd = d.dropna(subset=[mk])
        if dd["window"].nunique() < 2:
            f = f"{mk} ~ C(grp, Treatment('Control')) + C(strain) + C(sex)"
        else:
            f = f"{mk} ~ C(grp, Treatment('Control')) + C(strain) + C(sex) + C(window)"
        try:
            m = smf.ols(f, data=dd).fit()
        except Exception as e:
            print(f"  ! OLS failed for {mk}: {e}")
            continue
        ctrl_mean = dd[dd["grp"] == "Control"][mk].mean()
        rows = []
        for g in TREAT_GROUPS:
            nm = f"C(grp, Treatment('Control'))[T.{g}]"
            if nm in m.params:
                rows.append(dict(metric=mk, group=g, coef=m.params[nm], se=m.bse[nm], p_raw=m.pvalues[nm],
                                 pct_vs_ctrl=m.params[nm] / ctrl_mean * 100 if ctrl_mean else np.nan,
                                 n=int((dd["grp"] == g).sum())))
        r = pd.DataFrame(rows)
        if len(r):
            r["p_holm"] = holm(r["p_raw"])
            coefs.append(r)
        try:
            f2 = (f"{mk} ~ C(group) + C(strain)*C(sex) + C(group):C(strain) + C(group):C(sex)"
                  + (" + C(window)" if dd["window"].nunique() > 1 else ""))
            m2 = smf.ols(f2, data=dd).fit()
            a = sm.stats.anova_lm(m2, typ=2)
            a = a.reset_index().rename(columns={"index": "term"})
            a.insert(0, "metric", mk)
            anovas.append(a)
        except Exception as e:
            print(f"  ! ANOVA failed for {mk}: {e}")
    return (pd.concat(coefs, ignore_index=True) if coefs else pd.DataFrame(),
            pd.concat(anovas, ignore_index=True) if anovas else pd.DataFrame())


def stars(p):
    if p is None or np.isnan(p):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

### Figures helper functions

In [80]:
def save(fig, outdir, name):
    p = Path(outdir) / "figures"
    p.mkdir(parents=True, exist_ok=True)
    fig.savefig(p / f"{name}.png", bbox_inches="tight")
    fig.savefig(p / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {name}")


def group_title(g):
    return g.replace("-", " ")


def shade_night(ax, x0=0, x1=24, day=DAY_HOURS):
    ax.axvspan(day, x1, color=NIGHT, alpha=0.55, lw=0, zorder=0)


def fig_overview(data, light, wobj, lights_on, outdir):
    mt = data[ACTIVITY if ACTIVITY in data else "MT"]
    full = pd.date_range(mt.index.min(), mt.index.max(), freq="min")
    mean_act = mt.mean(axis=1).reindex(full)
    smooth = mean_act.rolling(30, center=True, min_periods=10).mean()
    lt = light.reindex(full)
    fig, (ax0, ax) = plt.subplots(2, 1, figsize=(15, 5.2), sharex=True, gridspec_kw=dict(height_ratios=[0.16, 1], hspace=0.04))
    # light strip
    xs = mpl.dates.date2num(full.to_pydatetime())
    ax0.pcolormesh(np.append(xs, xs[-1] + 1 / 1440), [0, 1], np.where(np.isnan(lt.values), np.nan, lt.values)[None, :],
                   cmap=LinearSegmentedColormap.from_list("ld", ["#2b2d42", "#ffe08a"]), vmin=0, vmax=1)
    ax0.set_facecolor("#eeeeee")
    ax0.set_yticks([0.5]); ax0.set_yticklabels(["light"], fontsize=8.5)
    ax0.grid(False); ax0.tick_params(length=0)
    for s in ax0.spines.values():
        s.set_visible(False)
    ax.plot(full, smooth.values, color="#08519c", lw=1.2)
    ymax = np.nanmax(smooth.values) * 1.18
    for wn, wo in wobj.items():
        ax.axvspan(wo["idx"][0], wo["idx"][-1], color=WIN_COLORS[wn], alpha=0.10, lw=0)
        ax.text(wo["idx"][0] + (wo["idx"][-1] - wo["idx"][0]) / 2, ymax * 0.97, f"{wn}  ({wo['regime']})", ha="center", va="top",
                fontweight="bold", color=WIN_COLORS[wn])
    gaps = mean_act.isna()
    ax.fill_between(full, 0, ymax, where=gaps.values, color="#999999", alpha=0.35, lw=0, step="mid")
    ax.set_ylim(0, ymax)
    ax.set_ylabel("Mean moves / min, all tubes\n(30-min running mean)")
    ax.xaxis.set_major_locator(mpl.dates.HourLocator(byhour=range(0, 24, 6)))
    ax.xaxis.set_major_formatter(mpl.dates.DateFormatter("%d %b\n%H:%M"))
    ax.set_title("Experiment overview: monitor 1, all valid minutes", loc="left")
    ax.text(0.995, 0.02, "grey = no data (monitor restarting)   |   shaded = analysis windows", transform=ax.transAxes,
            ha="right", color=MUTED, fontsize=8.5)
    save(fig, outdir, "00_overview")


def fig_window_heatmap(wname, wo, fl, outdir, lights_on):
    d = fl[fl["window"] == wname].sort_values("tube")
    act = wo["act"][d["tube"].tolist()].to_numpy().T          # tubes x minutes
    n = len(d)
    fig = plt.figure(figsize=(14, 0.3 * n + 2.4))
    gs = GridSpec(2, 2, height_ratios=[0.5, n], width_ratios=[30, 3], hspace=0.04, wspace=0.03, figure=fig)
    axl, ax, axb = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    hrs = len(wo["idx"]) / 60
    cm = SEQ.copy(); cm.set_bad("#bdbdbd")
    vmax = np.nanpercentile(act, 99.5)
    im = ax.imshow(np.sqrt(act), aspect="auto", cmap=cm, vmin=0, vmax=np.sqrt(vmax), extent=[0, hrs, n, 0],
                   interpolation="nearest")
    ax.grid(False)
    ax.set_yticks(np.arange(n) + 0.5)
    ylab = [f"{r.tube:02d}  {r.label}" for r in d.itertuples()]
    ax.set_yticklabels(ylab, fontsize=8)
    for t, r in zip(ax.get_yticklabels(), d.itertuples()):
        if r.excluded:
            t.set_color("#c0392b")
    for i, r in enumerate(d.itertuples()):
        if r.excluded:
            ax.add_patch(plt.Rectangle((0, i), hrs, 1, fill=False, ec="#c0392b", lw=0.8, hatch="////", alpha=0.6))
    # light strip from sensor
    lt = wo["light"].to_numpy()[None, :]
    axl.imshow(lt, aspect="auto", cmap=LinearSegmentedColormap.from_list("ld", ["#2b2d42", "#ffe08a"]), vmin=0, vmax=1,
               extent=[0, hrs, 0, 1], interpolation="nearest")
    axl.set_yticks([0.5]); axl.set_yticklabels(["light"], fontsize=8); axl.set_xticks([]); axl.grid(False)
    for s in axl.spines.values():
        s.set_visible(False)
    # clock ticks
    t0 = wo["start"]
    ticks = np.arange(0, hrs + 0.01, 2)
    ax.set_xticks(ticks)
    ax.set_xticklabels([(t0 + pd.Timedelta(hours=float(h))).strftime("%H:%M") for h in ticks], fontsize=8.5)
    ax.set_xlabel(f"Clock time  (window start {t0:%d %b %H:%M})")
    # lights on/off vertical dashes (from ZT)
    zt = wo["zt"]
    for i in np.where(np.diff(np.floor(zt / DAY_HOURS)) != 0)[0]:
        ax.axvline((i + 1) / 60, color="#222", lw=0.8, ls="--", alpha=0.6)
    # side bar: sleep %
    sp = (d["sleep_min"] / (d["valid_frac"] * len(wo["idx"])).clip(lower=1) * 100).to_numpy()
    cols = ["#c9c9c9" if e else "#3b8ac4" for e in d["excluded"]]
    axb.barh(np.arange(n) + 0.5, sp, color=cols, height=0.7)
    axb.set_ylim(n, 0); axb.set_yticks([]); axb.set_xlim(0, 100)
    axb.set_xlabel("sleep %"); axb.grid(axis="x"); axb.grid(axis="y", visible=False)
    cax = fig.add_axes([0.905, 0.92, 0.09, 0.012])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_ticks([0, np.sqrt(vmax)]); cb.set_ticklabels(["0", f"{vmax:.0f}"]); cb.ax.tick_params(labelsize=7.5)
    cax.set_title("moves / min (sqrt scale)", fontsize=7.5, loc="left")
    axl.set_title(f"{wname}  ·  {wo['regime']}  ·  per-tube activity  (red hatch = excluded)", loc="left")
    save(fig, outdir, f"01_heatmap_{wname}")


def _group_lines(ax, prof_df, groups, ylabel=None, lw=1.6):
    x = (np.arange(prof_df.shape[1]) + 0.5) * BIN_MIN / 60 if False else None
    return


def profile_matrix(fl, prof, which):
    """-> DataFrame fly x bin"""
    k = 0 if which == "act" else 1
    return pd.DataFrame({fid: prof[fid][k] for fid in fl["fid"]}).T


def fig_profiles(fl, prof, which, outdir, name, title, regime="LD", strata=None, ylabel=""):
    d = fl[(~fl["excluded"]) & (fl["regime"] == regime)]
    if len(d) == 0:
        return
    pm = profile_matrix(d, prof, which)
    x = (np.arange(pm.shape[1]) + 0.5) * BIN_MIN / 60
    if strata is None:
        strata = [(f"{STRAIN_NAME[a]} {SEX_NAME[b]}", (d["strain"] == a) & (d["sex"] == b)) for a, b in STRATA]
    fig, axes = plt.subplots(len(strata), 2, figsize=(12.5, 2.7 * len(strata) + 0.8), sharex=True, sharey="row",
                             squeeze=False)
    for r, (st_title, mask) in enumerate(strata):
        ymax = 0
        for c, trt in enumerate(["IG", "TG"]):
            ax = axes[r, c]
            shade_night(ax)
            for g in CONTROL_GROUPS + [g for g in TREAT_GROUPS if g.startswith(trt)]:
                ids = d.loc[mask & (d["group"] == g), "fid"]
                if len(ids) == 0:
                    continue
                sub = pm.loc[ids]
                mu, se = sub.mean(), sub.sem() if len(sub) > 1 else sub.mean() * 0
                is_c = g in CONTROL_GROUPS
                ax.plot(x, mu.values, color=GROUP_COLORS[g], lw=1.2 if is_c else 1.8, ls="--" if is_c else "-",
                        label=f"{group_title(g)} (n={len(ids)})", zorder=3)
                ax.fill_between(x, (mu - se).values, (mu + se).values, color=GROUP_COLORS[g], alpha=0.14, lw=0, zorder=2)
                ymax = max(ymax, np.nanmax((mu + se).values))
            if r == 0:
                ax.set_title(f"{trt} vs controls")
            ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 3))
            ax.legend(loc="upper right", fontsize=7.5, ncol=1, handlelength=1.6, borderaxespad=0.2)
            if c == 0:
                ax.set_ylabel(f"{st_title}\n{ylabel}", fontsize=9)
        axes[r, 0].set_ylim(0, ymax * 1.12)
    axes[-1, 0].set_xlabel("Zeitgeber time (h)  ·  shaded = night" if regime == "LD" else "Circadian time (h)  ·  shaded = subjective night")
    axes[-1, 1].set_xlabel(axes[-1, 0].get_xlabel())
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=12, y=1.0)
    fig.tight_layout()
    save(fig, outdir, name)


def fig_metric_grid(fl, metrics, outdir, name, title, st):
    d = fl[~fl["excluded"]]
    pos = dict(zip(GROUP_ORDER, [0, 1, 2.7, 3.7, 4.7, 6.4, 7.4, 8.4]))
    fig, axes = plt.subplots(len(metrics), 4, figsize=(15, 2.5 * len(metrics) + 1), squeeze=False)
    rng = np.random.default_rng(1)
    for r, mk in enumerate(metrics):
        for c, (sa, sx) in enumerate(STRATA):
            ax = axes[r, c]
            sub = d[(d["strain"] == sa) & (d["sex"] == sx)]
            ymax = sub[mk].max() if sub[mk].notna().any() else 1
            for g in GROUP_ORDER:
                v = sub[sub["group"] == g]
                if v.empty:
                    continue
                for w, vv in v.groupby("window"):
                    ax.scatter(pos[g] + rng.uniform(-0.22, 0.22, len(vv)), vv[mk], s=24, marker=WIN_MARK[w],
                               color=GROUP_COLORS[g], edgecolor="white", linewidth=0.5, alpha=0.95, zorder=3)
                m, se = v[mk].mean(), v[mk].sem() if v[mk].notna().sum() > 1 else 0
                ax.plot([pos[g] - 0.34, pos[g] + 0.34], [m, m], color=INK, lw=2, zorder=4, solid_capstyle="butt")
                ax.errorbar(pos[g], m, yerr=se, color=INK, lw=1, capsize=2.5, zorder=4)
            # stars (stratum, unadjusted Mann-Whitney vs pooled controls)
            srow = st[(st["metric"] == mk) & (st["stratum"] == f"{sa}-{sx}")].set_index("group")
            for g in TREAT_GROUPS:
                if g in srow.index:
                    s = stars(srow.loc[g, "p_raw"])
                    if s:
                        ax.text(pos[g], ymax * 1.04, s, ha="center", va="bottom", fontsize=10, color=INK, fontweight="bold")
            ax.set_ylim(0 if mk != "lat_night" else 0, ymax * 1.16 if ymax > 0 else 1)
            ax.set_xlim(-0.7, 9.1)
            ax.set_xticks([pos[g] for g in GROUP_ORDER])
            ax.set_xticklabels(["C1", "C2", "Lo", "Med", "Hi", "Lo", "Med", "Hi"] if r == len(metrics) - 1 else [],
                               fontsize=8.5)
            ax.grid(axis="x", visible=False)
            if r == 0:
                ax.set_title(f"{STRAIN_NAME[sa]} {SEX_NAME[sx]}")
            if c == 0:
                ax.set_ylabel(METRICS[mk][0], fontsize=8.5)
            if r == len(metrics) - 1:
                ax.text(pos["IG-Med"], -0.30, "IG", transform=ax.get_xaxis_transform(), ha="center", fontweight="bold", color=GROUP_COLORS["IG-Hi"])
                ax.text(pos["TG-Med"], -0.30, "TG", transform=ax.get_xaxis_transform(), ha="center", fontweight="bold", color=GROUP_COLORS["TG-Hi"])
                ax.text(0.5, -0.30, "Controls", transform=ax.get_xaxis_transform(), ha="center", fontweight="bold", color=MUTED)
    handles = [Line2D([], [], marker=WIN_MARK[w], ls="", color="#777", markersize=6, label=w) for w in WIN_MARK]
    handles.append(Line2D([], [], color=INK, lw=2, label="mean ± SEM"))
    fig.legend(handles=handles, loc="upper right", ncol=5, bbox_to_anchor=(0.99, 1.0), fontsize=8.5)
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=12, y=1.0)
    fig.text(0.01, -0.005, "* p<0.05, ** p<0.01 vs pooled controls of the same strain & sex (Mann-Whitney, uncorrected, small n - exploratory).  "
             "Each dot = one fly; marker = run window.", color=MUTED, fontsize=8.5, ha="left")
    fig.tight_layout()
    save(fig, outdir, name)


def fig_effects(fl, st, ols, metrics, outdir):
    d = fl[~fl["excluded"]]
    cols = ["all"] + [f"{a}-{b}" for a, b in STRATA]
    clab = ["Pooled"] + [f"{STRAIN_NAME[a]}\n{SEX_NAME[b]}" for a, b in STRATA]
    nr = 2
    nc = int(np.ceil(len(metrics) / nr))
    fig, axes = plt.subplots(nr, nc, figsize=(4.6 * nc, 3.6 * nr + 0.6), squeeze=False)
    for ax, mk in zip(axes.ravel(), metrics):
        M = np.full((len(TREAT_GROUPS), len(cols)), np.nan)
        S = np.full(M.shape, "", dtype=object)
        for i, g in enumerate(TREAT_GROUPS):
            for j, cc in enumerate(cols):
                row = st[(st["metric"] == mk) & (st["stratum"] == cc) & (st["group"] == g)]
                if len(row):
                    M[i, j] = row["pct_change"].iloc[0]
                    S[i, j] = stars(row["p_raw"].iloc[0])
            if len(ols):
                o = ols[(ols["metric"] == mk) & (ols["group"] == g)]
                if len(o):
                    S[i, 0] = stars(o["p_holm"].iloc[0])
        ax.grid(False)
        im = ax.imshow(np.clip(M, -100, 100), cmap=DIV, vmin=-100, vmax=100, aspect="auto")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                if not np.isnan(M[i, j]):
                    v = M[i, j]
                    ax.text(j, i, f"{v:+.0f}%{S[i, j]}", ha="center", va="center", fontsize=7.8,
                            color="white" if abs(np.clip(v, -100, 100)) > 62 else INK)
        ax.set_xticks(range(len(cols))); ax.set_xticklabels(clab, fontsize=8)
        ax.set_yticks(range(len(TREAT_GROUPS))); ax.set_yticklabels([group_title(g) for g in TREAT_GROUPS], fontsize=8.5)
        for k in (0.5,):
            ax.axvline(k, color="white", lw=3)
        ax.axhline(2.5, color="white", lw=3)
        ax.set_title(METRICS[mk][1])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.tick_params(length=0)
    for ax in axes.ravel()[len(metrics):]:
        ax.axis("off")
    cax = fig.add_axes([0.35, -0.01, 0.3, 0.016])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label("% change vs pooled controls (clipped at ±100%)")
    fig.suptitle("Effect sizes: each treatment group relative to pooled controls", x=0.01, ha="left", fontweight="bold",
                 fontsize=12, y=1.0)
    fig.text(0.01, -0.02, "Pooled column: OLS (strain + sex + window adjusted), Holm-corrected across 6 groups.  Stratum columns: Mann-Whitney, uncorrected.\n"
             "* p<0.05  ** p<0.01  *** p<0.001", color=MUTED, fontsize=8.5)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, outdir, "05_effect_heatmap")


def fig_bout_ecdf(fl, bouts, outdir):
    d = fl[(~fl["excluded"]) & (fl["regime"] == "LD")]
    b = bouts[bouts["fid"].isin(d["fid"])].merge(d[["fid", "group"]], on="fid")
    if b.empty:
        return
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), sharex=True, sharey=True)
    for r, (lab, isday) in enumerate([("Day (ZT0-12)", True), ("Night (ZT12-24)", False)]):
        for c, trt in enumerate(["IG", "TG"]):
            ax = axes[r, c]
            for g in CONTROL_GROUPS + [g for g in TREAT_GROUPS if g.startswith(trt)]:
                v = np.sort(b[(b["group"] == g) & (b["day"] == isday)]["length"].to_numpy())
                if len(v) == 0:
                    continue
                ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=GROUP_COLORS[g], lw=1.6,
                        ls="--" if g in CONTROL_GROUPS else "-", label=f"{group_title(g)} (n={len(v)} bouts)")
            ax.set_xscale("log"); ax.set_xlim(SLEEP_MIN, 800)
            ax.legend(loc="lower right", fontsize=7.5)
            ax.set_title(f"{trt} vs controls · {lab}")
            if c == 0:
                ax.set_ylabel("Cumulative fraction of bouts")
            if r == 1:
                ax.set_xlabel("Sleep-bout length (min, log)")
    fig.suptitle("Sleep-bout length distributions (LD windows, all strains & sexes pooled)", x=0.01, ha="left", fontweight="bold", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, outdir, "06_bout_ecdf")


def fig_position(fl, outdir):
    d = fl[~fl["excluded"]]
    pc = [f"pn{b}" for b in range(1, 16)]
    if pc[0] not in d:
        return
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.8), sharex=True)
    vmax = d.groupby(["stratum", "group"])[pc].mean().to_numpy().max()
    for ax, (sa, sx) in zip(axes.ravel(), STRATA):
        sub = d[(d["strain"] == sa) & (d["sex"] == sx)].groupby("group")[pc].mean().reindex(GROUP_ORDER)
        im = ax.imshow(sub.to_numpy() * 100, aspect="auto", cmap=SEQ, vmin=0, vmax=vmax * 100)
        ax.grid(False)
        ax.set_yticks(range(len(GROUP_ORDER))); ax.set_yticklabels([group_title(g) for g in GROUP_ORDER], fontsize=8.5)
        ax.set_xticks(range(15)); ax.set_xticklabels(range(1, 16), fontsize=8)
        ax.set_title(f"{STRAIN_NAME[sa]} {SEX_NAME[sx]}")
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
    for ax in axes[1]:
        ax.set_xlabel("Beam position (Pn, 1-15)")
    cax = fig.add_axes([0.35, -0.03, 0.3, 0.02])
    fig.colorbar(im, cax=cax, orientation="horizontal").set_label("% of minutes (end-of-minute position)")
    fig.suptitle("Where in the tube flies sit, by group", x=0.01, ha="left", fontweight="bold", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, outdir, "07_position")


def fig_pca(fl, outdir):
    d = fl[~fl["excluded"]].copy()
    keys = ["total_act", "waking_act", "act_day", "act_night", "sleep_day", "sleep_night", "n_bouts", "mean_bout",
            "max_bout", "lat_night"]
    d = d.dropna(subset=keys)
    if len(d) < 10:
        return
    X = d[keys].to_numpy(float)
    # remove strain x sex baselines so the treatment structure (not sex) drives the picture
    for s in d["stratum"].unique():
        m = (d["stratum"] == s).to_numpy()
        X[m] = X[m] - X[m].mean(0)
    X = X / X.std(0)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    sc = U * S
    ev = S ** 2 / (S ** 2).sum()
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw=dict(width_ratios=[1.5, 1]))
    for g in GROUP_ORDER:
        m = (d["group"] == g).to_numpy()
        ax.scatter(sc[m, 0], sc[m, 1], s=42, color=GROUP_COLORS[g], edgecolor="white", lw=0.6, label=group_title(g), zorder=3)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.0f}%)"); ax.set_ylabel(f"PC2 ({ev[1]*100:.0f}%)")
    ax.axhline(0, color="#bbb", lw=0.7); ax.axvline(0, color="#bbb", lw=0.7)
    ax.legend(ncol=2, fontsize=8, loc="best")
    ax.set_title("Flies in behaviour space (strain × sex baselines removed)")
    for i, k in enumerate(keys):
        ax2.arrow(0, 0, Vt[0, i], Vt[1, i], color="#555", head_width=0.025, length_includes_head=True)
        ax2.text(Vt[0, i] * 1.12, Vt[1, i] * 1.12, METRICS[k][1], fontsize=8, ha="center", va="center")
    ax2.set_xlim(-0.9, 0.9); ax2.set_ylim(-0.9, 0.9); ax2.set_aspect("equal")
    ax2.set_title("Loadings"); ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, outdir, "08_pca")


def fig_batch_qc(fl, outdir):
    d = fl[(~fl["excluded"]) & (fl["group"].isin(CONTROL_GROUPS))]
    mets = ["total_act", "sleep_min", "n_bouts", "mean_bout"]
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
    rng = np.random.default_rng(2)
    for ax, mk in zip(axes, mets):
        for i, (w, v) in enumerate(d.groupby("window")):
            for sx, col in (("M", "#3b6fb6"), ("F", "#c0504d")):
                vv = v[v["sex"] == sx][mk]
                ax.scatter(i + (-0.12 if sx == "M" else 0.12) + rng.uniform(-0.07, 0.07, len(vv)), vv, s=26, color=col,
                           edgecolor="white", lw=0.5, zorder=3)
            ax.plot([i - 0.3, i + 0.3], [v[mk].mean()] * 2, color=INK, lw=2, zorder=4)
        ax.set_xticks(range(d["window"].nunique()))
        ax.set_xticklabels([f"{w}\n{fl[fl.window == w]['regime'].iloc[0]}" for w in sorted(d["window"].unique())])
        ax.set_title(METRICS[mk][1]); ax.grid(axis="x", visible=False)
    axes[0].legend(handles=[Line2D([], [], marker="o", ls="", color="#3b6fb6", label="male"),
                            Line2D([], [], marker="o", ls="", color="#c0504d", label="female")], loc="upper left")
    fig.suptitle("Batch check: control flies only, by run window", x=0.01, ha="left", fontweight="bold", y=1.04)
    fig.tight_layout()
    save(fig, outdir, "09_batch_qc")

### Run Analysis

In [78]:
def run_analysis(datafile=DATA_FILE, outdir=OUTDIR):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)

    datafile = str(datafile)
    if not Path(datafile).exists():
        raise FileNotFoundError(
            f"Could not find the DAM data file: {datafile!r}. "
            "Update DATA_FILE in Cell 2 with the correct path."
        )
    print(f"Loading {datafile} ...")
    data, light, info = load_dam(datafile)
    print(f"  {info['n_rows']:,} rows ({info['n_bad']:,} with status != 1 -> ignored), types: {info['types']}")
    print(f"  span: {info['span'][0]}  ->  {info['span'][1]}")
    lo, ons, offs = estimate_lights_on(light)
    if LIGHTS_ON_CLOCK:
        h, m = map(int, LIGHTS_ON_CLOCK.split(":")); lo = h * 60 + m
    if lo is None:
        lo = 9 * 60
        print("  ! no light transitions found, assuming lights-on 09:00")
    print(f"  lights-on (ZT0) = {int(lo // 60):02d}:{int(lo % 60):02d}   "
          f"(transitions on: {[t.strftime('%d %H:%M') for t in ons]}, off: {[t.strftime('%d %H:%M') for t in offs]})")

    print("Processing windows ...")
    fl, bouts, prof, wobj = process_windows(data, light, lo)
    for wn, wo in wobj.items():
        print(f"  {wn}: {wo['regime']}, {len(wo['idx'])} min, {wo['valid_frac']*100:.1f}% with data")
    fl.to_csv(out / "per_fly_metrics.csv", index=False)
    bouts.to_csv(out / "sleep_bouts.csv", index=False)
    qc = fl[fl["excluded"] | fl["possibly_dead"]][["fid", "window", "tube", "label", "valid_frac", "total_act",
                                                    "hours_since_last_move", "flags"]]
    qc.to_csv(out / "qc_flags.csv", index=False)
    print(f"  {len(fl)} tubes, {int(fl['excluded'].sum())} excluded:")
    for r in qc.itertuples():
        print(f"    {r.fid} {r.label:<13} {r.flags or '(flagged only)'}")

    # design coverage
    allcells = [f"{t}{d}{s}{x}" for t in ("IG", "TG") for d in ("Lo", "Med", "Hi") for s in ("YW", "OrR") for x in "MF"]
    allcells += [f"{c}{s}{x}" for c in CONTROL_GROUPS for s in ("YW", "OrR") for x in "MF"]
    for wn, w in WINDOWS.items():
        miss = sorted(set(allcells) - set(w["labels"]))
        if miss:
            print(f"  {wn}: design cells not in label list: {miss}")

    d = fl[~fl["excluded"]]
    if STAT_WINDOWS:
        d = d[d["window"].isin(STAT_WINDOWS)]
    print(f"Analysing {len(d)} flies.")
    # summary table by group
    summ = d.groupby(["stratum", "group"])[list(METRICS)].agg(["mean", "sem", "count"])
    summ.to_csv(out / "group_summary.csv")

    mkeys = list(METRICS)
    print("Statistics ...")
    st = stratum_tests(d, mkeys)
    st.to_csv(out / "stats_vs_control_by_stratum.csv", index=False)
    ols, anova = ols_tests(d, mkeys)
    if len(ols):
        ols.to_csv(out / "stats_vs_control_pooled_OLS.csv", index=False)
    if len(anova):
        anova.to_csv(out / "stats_anova_typeII.csv", index=False)

    print("Figures ...")
    fig_overview(data, light, wobj, lo, out)
    for wn, wo in wobj.items():
        fig_window_heatmap(wn, wo, fl, out, lo)
    dfl = fl.copy()
    dfl.loc[~dfl["fid"].isin(d["fid"]), "excluded"] = True
    fig_profiles(dfl, prof, "act", out, "02_activity_profile_LD", "Activity profile (LD windows)", "LD",
                 ylabel=f"{ACTIVITY} per {BIN_MIN} min")
    fig_profiles(dfl, prof, "sleep", out, "03_sleep_profile_LD", "Sleep profile (LD windows)", "LD",
                 ylabel=f"sleep (min per {BIN_MIN} min)")
    if (dfl["regime"] == "DD").any():
        dd_strata = [("all strains & sexes", dfl["regime"] == "DD")]
        fig_profiles(dfl, prof, "act", out, "02b_activity_profile_DD", "Activity profile (constant darkness)", "DD",
                     strata=dd_strata, ylabel=f"{ACTIVITY} per {BIN_MIN} min")
        fig_profiles(dfl, prof, "sleep", out, "03b_sleep_profile_DD", "Sleep profile (constant darkness)", "DD",
                     strata=dd_strata, ylabel=f"sleep (min per {BIN_MIN} min)")
    fig_metric_grid(d, ["total_act", "waking_act", "sleep_min", "sleep_night"], out, "04a_metrics_activity_sleep",
                    "Activity and sleep amount by group", st)
    fig_metric_grid(d, ["n_bouts", "mean_bout", "max_bout", "lat_night"], out, "04b_metrics_sleep_architecture",
                    "Sleep architecture by group", st)
    fig_effects(d, st, ols, ["total_act", "waking_act", "sleep_min", "sleep_night", "n_bouts", "mean_bout"], out)
    fig_bout_ecdf(dfl, bouts, out)
    fig_position(d, out)
    fig_pca(d, out)
    fig_batch_qc(d, out)
    print(f"\nDone. Results in {out.resolve()}")

In [79]:
run_analysis(DATA_FILE, OUTDIR)

Loading DAM Monitor 1 Oct 3-8.txt ...
  49,396 rows (1,516 with status != 1 -> ignored), types: ['CT', 'D12', 'D34', 'L12', 'L34', 'MT', 'Pn', 'Rt']
  span: 2026-10-03 12:25:00  ->  2026-10-08 17:25:00
  lights-on (ZT0) = 09:16   (transitions on: ['06 09:17', '07 09:16', '08 09:08'], off: ['05 20:59', '06 21:17', '07 21:09'])
Processing windows ...
  W1: DD, 1440 min, 99.9% with data
  W2: LD, 1440 min, 99.9% with data
  W3: LD, 1440 min, 99.4% with data
  W4: LD, 1440 min, 99.9% with data
  128 tubes, 10 excluded:
    W1-T16 TGLoYWM       duplicate label (identity ambiguous)
    W1-T23 TGLoYWM       duplicate label (identity ambiguous)
    W1-T30 TGLoYWF       duplicate label (identity ambiguous)
    W1-T32 TGLoYWF       duplicate label (identity ambiguous)
    W2-T04 IGLoYWF       no movement for last 24 h (dead?)
    W2-T06 IGHiYWM       duplicate label (identity ambiguous)
    W2-T13 IGHiYWM       duplicate label (identity ambiguous)
    W3-T14 TGLoYWM       no movement for last 8 

C:\Users\dell\AppData\Local\Temp\ipykernel_19128\88995839.py:253: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  saved 05_effect_heatmap
  saved 06_bout_ecdf


C:\Users\dell\AppData\Local\Temp\ipykernel_19128\88995839.py:306: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  saved 07_position
  saved 08_pca
  saved 09_batch_qc

Done. Results in C:\Users\dell\Yim Lab\dam_results
